In [0]:

#Leyendo la tabla de caracteristicas generales
ventas = spark.read.format("delta").table("workspace.electiva.caso_practico_ventas")

# Convertir el DataFrame de Spark a Pandas
ventas = ventas.toPandas()

ventas

In [0]:
import pandas as pd

# 2. Identificación de Tipos
print("\n--- Tipos de datos asignados ---")
print(ventas.dtypes)

# 3. Detección de Nulos
print("\n--- Resumen de Valores Nulos ---")
nulos_total = ventas.isnull().sum()
nulos_pct = (ventas.isnull().mean() * 100).round(2)
print(pd.DataFrame({'Total Nulos': nulos_total, 'Porcentaje (%)': nulos_pct}))

# 4. Estadística Descriptiva (solo variables numéricas principales)
print("\n--- Estadística Descriptiva ---")
print(ventas[['cantidad', 'precio_unitario']].describe())

Fase 2 - Limpieza

In [0]:
ventas = ventas.drop_duplicates(subset=['id_transaccion'])

ventas['metodo_pago'] = ventas['metodo_pago'].replace({'Tarjeta Credito': 'Tarjeta de Crédito'})

mediana = ventas['cantidad'].median()
ventas['cantidad'] = ventas['cantidad'].fillna(mediana)

moda = ventas['satisfaccion_cliente'].mode()[0]
ventas['satisfaccion_cliente'] = ventas['satisfaccion_cliente'].fillna(moda)

filtro_outlier = ventas[ventas['precio_unitario'] == 9999].index
ventas = ventas.drop(filtro_outlier)



Fase 3 - Transformación e Ingeniería de Atributos

In [0]:
import pandas as pd

ventas['monto_total'] = ventas['cantidad'] * ventas['precio_unitario']

ventas['fecha'] = pd.to_datetime(ventas['fecha'])
ventas['dia_semana'] = ventas['fecha'].dt.day_name()
ventas['mes'] = ventas['fecha'].dt.month

ventas['rango_monto'] = pd.cut(ventas['monto_total'], bins=[0, 100, 500, 999999], labels=['Bajo', 'Medio', 'Alto'])

Fase 4: Agregación y Resumen de Datos

In [0]:
resumen_productos = ventas.groupby('categoria_producto').agg({'monto_total': 'sum', 'cantidad': 'mean'})

matriz = pd.pivot_table(ventas, values='monto_total', index='region', columns='categoria_producto')

satisfaccion = ventas.groupby('metodo_pago')['satisfaccion_cliente'].mean()

Fase 5: Selección, Filtrado y Muestreo

In [0]:
# 1. Filtrado Multicriterio
# Creamos una máscara booleana con las dos condiciones solicitadas
filtro_norte = (ventas['region'] == 'Norte') 
filtro_satisfaccion = (ventas['satisfaccion_cliente'] >= 4)

# Aplicamos los filtros a la tabla original usando el operador & (Y)
ventas_norte_exitosas = ventas[filtro_norte & filtro_satisfaccion]

# 2. Muestreo Estratificado
# Extraemos el 10% de los datos totales, pero agrupados por región para no perder la proporción
muestra = ventas.groupby('region').sample(frac=0.10)